# Aegis Command - Zero-Shot Model Bake-Off (Colab GPU Pipeline)

This notebook runs the **Zero-Shot Model Bake-Off Matrix** on Google Colab (T4 / V100 / A100 GPU) across all 155 human-verified aerial frames.

> **VERIFICATION PROTOCOL NOTICE**:
> Per project truth hierarchy, this notebook **ONLY dumps raw predictions and execution metadata** (`predictions.json` and `run_meta.json` at candidate confidence $\ge 0.001$).
> All accuracy metrics, AP50, bootstrap confidence intervals, and comparison tables are **computed locally** on the workstation using `python3 scripts/compute_tables.py` and validated by `scripts/verify_benchmarks.py`.

### Ground Truth Certification:
- **Total Frames**: 155 frames extracted from `vision/source/drone_feed.mp4`
- **Certified Marker**: `vision/eval/labels/VERIFIED`
- **SHA256**: `a7fb1542fdb1e911299bb73971abc3abeea4aed675ef2a172933a4b168f595fc`
- **Verified Boxes**: 1,948 total (Person: $n=1,737$, Truck: $n=153$, Car: $n=58$, Bus: $n=0$)

### Configurations Executed:
- Pretrained models: `yolov8n`, `yolov8s`, `yolov8m`, `yolo11n`, `yolo11s`, `yolo11m`
- Resolutions: 640px, 960px, 1280px
- SAHI: `yolov8s` with 640x640 slice windows, 0.2 overlap ratio

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Dependencies
%pip install -q ultralytics sahi albumentations opencv-python matplotlib pandas numpy tabulate

In [ ]:
# 3. Clone Aegis Command Repository & Checkout Branch
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/accuracy-v2
!git pull

In [ ]:
# 4. Frame Verification & Regeneration (if needed)
from pathlib import Path
import os
from vision.eval.sample_frames import sample_frames

images_dir = Path("vision/eval/images")
existing_images = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))

if len(existing_images) < 155:
    print(f"Found {len(existing_images)} frames. Regenerating 155 frames from vision/source/drone_feed.mp4...")
    sample_frames(
        video_path="vision/source/drone_feed.mp4",
        out_dir="vision/eval/images",
        manifest_path="vision/eval/manifest.json",
        stride=13,
        offset=6
    )
else:
    print(f"Verified: {len(existing_images)} frames exist in {images_dir}.")

In [ ]:
# 5. Cryptographic Certification Gate Validation
from vision.eval.eval_detection import check_verification_gate

labels_path = Path("vision/eval/labels")
marker = check_verification_gate(labels_path)
print(f"[GATE PASSED] Ground truth cryptographically certified at {marker}")

In [ ]:
# 6. Execute Zero-Shot Matrix Inference (Dumps Raw Predictions)
from vision.eval.eval_detection import run_evaluation

CONFIGURATIONS = [
    ("yolov8n.pt", 640),
    ("yolov8n.pt", 960),
    ("yolov8n.pt", 1280),
    ("yolov8s.pt", 640),
    ("yolov8s.pt", 960),
    ("yolov8s.pt", 1280),
    ("yolov8m.pt", 640),
    ("yolov8m.pt", 960),
    ("yolov8m.pt", 1280),
    ("yolo11n.pt", 640),
    ("yolo11n.pt", 960),
    ("yolo11n.pt", 1280),
    ("yolo11s.pt", 640),
    ("yolo11s.pt", 960),
    ("yolo11s.pt", 1280),
    ("yolo11m.pt", 1280),
]

for model_name, imgsz in CONFIGURATIONS:
    stem = Path(model_name).stem
    run_id = f"{stem}_{imgsz}px_all"
    print(f"\n{'='*70}")
    print(f"RUNNING INFERENCE: {model_name} @ {imgsz}px on CUDA (Run ID: {run_id})")
    print(f"{'='*70}")
    run_evaluation(
        images_dir="vision/eval/images",
        labels_dir="vision/eval/labels",
        model_path=model_name,
        imgsz=imgsz,
        device="cuda:0",
        save_raw=True,
        raw_root="docs/results/raw",
        run_id=run_id
    )

In [ ]:
# 7. SAHI (Sliced Aided Hyper Inference) GPU Raw Dump
import time
import json
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from vision.eval.finalize_labels import compute_labels_sha256

print("Running SAHI Sliced Inference with yolov8s (slice=640x640, overlap=0.2)...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path="yolov8s.pt",
    confidence_threshold=0.001,
    device="cuda:0"
)

images = sorted(list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png")))
sahi_raw_predictions = []
latencies_ms = []

t_start = time.perf_counter()
COCO_TO_EVAL = {0: 0, 2: 1, 7: 2}

for img in images:
    t0 = time.perf_counter()
    result = get_sliced_prediction(
        str(img),
        detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

    for obj in result.object_prediction_list:
        cid = obj.category.id
        if cid in COCO_TO_EVAL:
            bbox = obj.bbox.to_xyxy()
            x1, y1, x2, y2 = bbox[0] / 1920.0, bbox[1] / 1080.0, bbox[2] / 1920.0, bbox[3] / 1080.0
            sahi_raw_predictions.append({
                "frame_id": img.stem,
                "image_id": img.stem,
                "class_id": COCO_TO_EVAL[cid],
                "class": COCO_TO_EVAL[cid],
                "conf": round(float(obj.score.value), 6),
                "box": [round(x1, 6), round(y1, 6), round(x2, 6), round(y2, 6)]
            })

total_wall = time.perf_counter() - t_start
sahi_dir = Path("docs/results/raw/sahi_yolov8s_640px_all")
sahi_dir.mkdir(parents=True, exist_ok=True)

sahi_meta = {
    "run_id": "sahi_yolov8s_640px_all",
    "command": "SAHI get_sliced_prediction (640x640 slices, overlap=0.2)",
    "device": "cuda:0",
    "imgsz": 640,
    "weights": "yolov8s.pt",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "wall_seconds": round(total_wall, 2),
    "labels_sha256": compute_labels_sha256(labels_path),
    "num_frames": len(images),
    "split": "all",
    "fold": 1,
    "latency_mean_ms": round(float(sum(latencies_ms) / len(latencies_ms)), 2),
    "latency_p50_ms": round(float(sorted(latencies_ms)[len(latencies_ms)//2]), 2)
}

with open(sahi_dir / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(sahi_raw_predictions, f, indent=2)
with open(sahi_dir / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(sahi_meta, f, indent=2)

print(f"Saved {len(sahi_raw_predictions)} SAHI predictions to {sahi_dir}")

In [ ]:
# 8. Package Raw Output Artifacts for Download
!zip -r colab_bakeoff_raw.zip docs/results/raw/

from google.colab import files
files.download("colab_bakeoff_raw.zip")
print("\n[DOWNLOAD READY] Extract colab_bakeoff_raw.zip into repo docs/results/raw/ and run scripts/compute_tables.py locally.")